# Week 3 Lecture: Become a Data Detective 🔍

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-03-lecture.ipynb)

Every analysis starts the same way: someone hands you a file and a question. Before you can answer anything, you have to get that data into Python, figure out what's actually in it, and narrow it down to the rows and columns that matter. That's the whole job this week.

We'll work with the **Ames, Iowa housing dataset** — 2,930 home sales and 82 columns of detail about each one.

## How to use this notebook

This notebook accompanies the Week 3 Tuesday lecture. You can:

- **Follow along during class** — run each cell as we discuss it
- **Pause and experiment** — change the code and see what happens
- **Reference after class** — use it alongside the textbook chapters

## This Week's Topics

- **Chapter 7: Importing Data** — getting files off disk (or the web) and into Python
- **Chapter 8: DataFrames** — the difference between a DataFrame and a Series, and why it matters
- **Chapter 9: Subsetting** — selecting columns, filtering rows, and doing both at once

## Setup

In [ ]:
import pandas as pd

# Reading from a URL works everywhere -- your laptop or Colab
url = "https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_raw.csv"
ames = pd.read_csv(url)

print(f"Loaded {ames.shape[0]:,} home sales with {ames.shape[1]} columns")

## Getting Data Into Python

Data sits on a **disk** (your hard drive, or a server somewhere). Python can't work with it there — `read_csv()` copies it into **memory** so you can analyze it fast.

> **Copied, not connected.** Importing makes a *copy*. It does not open a live link to the file.
>
> This matters most in **Colab**, where your session runs on a temporary cloud machine — close the notebook or restart the runtime and that copy is gone. Same idea locally: close Python and `ames` disappears.
>
> **The takeaway:** every session starts by re-importing your data. That's not a chore — it's what makes your analysis reproducible.

### Reading from a local file

Above we read from a URL. `read_csv()` takes a **file path** just as happily:

```python
# Absolute path -- the full address from the root of your computer
pd.read_csv("/Users/jane/project/data/ames_raw.csv")

# Relative path -- directions from where you are right now
pd.read_csv("../data/ames_raw.csv")
```

Picture the project this way:

```
my_project/
├── notebooks/
│   └── analysis.ipynb  ← You are here
└── data/
    └── ames_raw.csv    ← Your data
```

`..` means "go up one folder." **Prefer relative paths** — they still work when you share the project with a teammate whose folders look nothing like yours.

### Uploading a file in Colab

If your data isn't online and isn't on the Colab machine, you can upload it by hand:

```python
from google.colab import files
uploaded = files.upload()          # opens a file picker
ames = pd.read_csv('ames_raw.csv') # then read it like any local file
```

Files uploaded this way **only last for the current session**.

## Getting to Know Your Data 🧐

Here's what we just imported. Take a look at the shape of it — rows down the side, columns across the top.

In [ ]:
ames

### Size It Up Yourself

Pandas gives you a whole toolkit for getting to know a new dataset. Run each of these and **read what comes back** — don't just execute them.

In [ ]:
ames.shape       # How big is it?

In [ ]:
ames.columns     # What are the columns called?

In [ ]:
ames.dtypes      # What type is each column?

In [ ]:
ames.head()      # What do the first rows look like?

In [ ]:
ames.info()      # Types and missing values, all at once

In [ ]:
ames.describe()  # Summary statistics for numeric columns

### Attributes vs Methods

Did you notice that some of those had parentheses and some didn't?

| | Examples | What it is |
|---|---|---|
| **Attributes** | `.shape`, `.columns`, `.dtypes` | A stored *property* of the object — like reading its ID card. **No parentheses.** |
| **Methods** | `.head()`, `.info()`, `.describe()` | A *function* the object can run for you. **Always parentheses.** |

**Memory trick:** Methods = Actions = Parentheses.

Watch what happens when you forget them:

In [ ]:
# A method WITHOUT parentheses returns the function itself -- not your data
print(ames.head)

In [ ]:
# WITH parentheses, it actually runs
ames.head(3)

### Why Inspect Before You Analyze

Those six commands take about ten seconds. Skipping them costs hours.

What you're really checking:

- **`.shape`** — did every row load, or did the file get truncated?
- **`.dtypes`** — is `SalePrice` actually numeric, or did pandas read it as text because one cell had a stray character?
- **`.info()`** — which columns have missing values, and how many?
- **`.describe()`** — any impossible values? A minimum of 0, a maximum of 999999?

**The real point:** every dataset arrives with a story attached — *"this is last year's sales."* Inspection is how you check whether the data matches the story **before** you build an analysis on top of it.

> **Golden Rule:** `.shape`, `.info()`, `.describe()` — every time, before anything else.

### Try It

Use `.info()` and `.describe()` on `ames` to answer these:

- Which columns have **missing values**? Name two.
- What is the **oldest** year built in the dataset, and the **newest**?
- Does anything in `.describe()` look suspicious to you?

In [ ]:
# Your code here


## Understanding DataFrames & Series

A **DataFrame** is a table — two dimensional, with labeled rows (the index) and named columns. It's the Python equivalent of a spreadsheet.

Each individual column of that table is a **Series**.

That distinction sounds academic. It isn't — it changes what code will run.

### Quick Challenge 🎯

**Prediction game.** Before running anything, decide what each of these gives you — a Series or a DataFrame?

```python
ames['SalePrice']
ames[['SalePrice']]
ames[['SalePrice', 'Year Built']]
```

Make your prediction, *then* run the next three cells.

In [ ]:
ames['SalePrice'].head(3)          # single brackets

In [ ]:
ames[['SalePrice']].head(3)        # double brackets

In [ ]:
ames[['SalePrice', 'Year Built']].head(3)   # two columns

**What's actually happening:** the outer `[ ]` is pandas' selection syntax — the inner `[ ]` is just a plain Python **list**.

So `ames[['SalePrice']]` is really `ames[ ]` handed the list `['SalePrice']`. And because it's an ordinary list, you can keep adding to it — `['SalePrice', 'Year Built', 'Neighborhood']` — which is exactly why double brackets scale to as many columns as you want, and single brackets don't.

### Why Should You Care?

**Rule #1 — always know which one you're holding.** Luckily you can tell at a glance:

- A **Series** prints as one labeled column with a `dtype:` footer underneath and **no header row**. It has three parts: the **values**, the **index**, and a single **dtype**.
- A **DataFrame** prints as a **table with a header row** — even when it holds only one column.

Squint at the output: **header row = DataFrame**, **`dtype:` footer = Series**.

In [ ]:
print("--- Series ---")
print(ames['SalePrice'].head(3))

print("\n--- DataFrame ---")
print(ames[['SalePrice']].head(3))

**Rule #2 — they don't share the same toolkit.**

Sometimes the *same* attribute or method works on both but hands back **different output**:

In [ ]:
print("Series .shape:   ", ames['SalePrice'].shape)
print("DataFrame .shape:", ames[['SalePrice']].shape)

In [ ]:
ames['SalePrice'].mean()      # one number

In [ ]:
ames[['SalePrice']].mean()    # a Series -- note the dtype: footer!

A Series is 1-dimensional, so `.shape` is a single number and `.mean()` collapses to **one value**. A DataFrame reduces *column-wise*, so `.mean()` hands back a **Series** — one entry per column.

Other times an attribute exists for one and simply **isn't there** on the other:

```python
ames['SalePrice'].columns    # ❌ AttributeError -- a Series has no columns
ames[['SalePrice']].columns  # ✅ Index(['SalePrice'], dtype='object')
```

> **Key Rule:** Reach for `[[ ]]` when you want to keep working like a DataFrame — selecting several columns, chaining, joining, or writing back out to a file.

### Try It

Using the `ames` DataFrame:

- Extract the `Neighborhood` column as a **Series**. What neighborhood are the first 3 records in?
- Extract the `Neighborhood` column as a **DataFrame**.
- Extract `Neighborhood`, `Overall Qual`, and `SalePrice` together as a **DataFrame**.

In [ ]:
# Your code here


## Data Subsetting: Finding What Matters

Real analysis almost never uses the whole table. You narrow it down along **two dimensions**:

1. **Select columns** — *"I only care about price and year built"*
2. **Filter rows** — *"I only want houses built after 2000"*

Let's take them one at a time, then combine them.

### Selecting Columns

You already know how to do this from the Quick Challenge — one name in `[ ]` gives a Series, a list in `[[ ]]` gives a DataFrame. Here it is put to work:

In [ ]:
ames['SalePrice'].head(3)                                    # one column -> Series

In [ ]:
ames[['SalePrice', 'Year Built']].head(3)                    # two columns -> DataFrame

In [ ]:
ames[['SalePrice', 'Year Built', 'Neighborhood']].head(3)    # three columns -> DataFrame

### Data Mystery #1 🕵️

Using the Ames dataset, find:

1. How many houses are in our dataset?
2. What's the highest sale price?
3. What's the average year built?
4. How many **unique neighborhoods** are represented?

**Detective tools:** `.shape` · `.max()` · `.mean()` · `.nunique()`

**Hint:** Can't remember what a column is called? `ames.columns` will list every one of them.

In [ ]:
# Mystery 1: How many houses are in our dataset?
# Your code here:



In [ ]:
# Mystery 2: What's the highest sale price?
# Your code here:



In [ ]:
# Mystery 3: What's the average year built?
# Your code here:



In [ ]:
# Mystery 4: How many unique neighborhoods?
# Your code here:



### Filtering Rows: The Magic of Conditions

Filtering is a two-step idea. First you ask a yes/no question about **every row at once**, which gives you a column of `True`/`False`. Then you hand that column back to the DataFrame to keep only the `True` rows.

In [ ]:
# Step 1: create a condition -- True/False for each row
expensive_houses = ames['SalePrice'] > 200000
expensive_houses.head()

In [ ]:
# Step 2: use the condition to filter
filtered_ames = ames[expensive_houses]

print(f"Original dataset: {ames.shape[0]} houses")
print(f"Expensive houses: {filtered_ames.shape[0]} houses")

### Building More Complex Filters

Combine conditions with `&` (AND) and `|` (OR).

⚠️ **Always use `&` and `|`** with pandas — *not* the Python keywords `and` / `or`. And wrap each condition in parentheses.

**Question 1:** houses that are expensive **AND** recently built.

In [ ]:
# Step 1: define our conditions
expensive = ames['SalePrice'] > 200000
recent = ames['Year Built'] > 2000

# Step 2: combine with & (AND)
expensive_and_recent = expensive & recent

# Step 3: filter the data
result = ames[expensive_and_recent]
print(f"Expensive AND recent houses: {result.shape[0]}")

**Question 2:** houses at either end of the market — inexpensive (under \$100,000) **OR** expensive (over \$500,000).

In [ ]:
# Step 1: define our conditions
inexpensive = ames['SalePrice'] < 100000
expensive = ames['SalePrice'] > 500000

# Step 2: combine with | (OR)
either_extreme = inexpensive | expensive

# Step 3: filter the data
result = ames[either_extreme]
print(f"Inexpensive: {inexpensive.sum()}  |  Expensive: {expensive.sum()}")
print(f"Either extreme: {result.shape[0]}")

Notice the difference: `&` **narrows** your results — a row must satisfy *both* conditions. `|` **widens** them — a row only needs *one*. Here no house can be under \$100k and over \$500k at once, so the two groups don't overlap and the counts simply add up.

### The Powerful `.loc` Accessor

So far we've treated our two dimensions separately — pick columns, *or* filter rows. In practice you almost always want them at the same time:

> *"Give me the neighborhood, price, and size — but only for the homes I actually care about."*

`.loc[]` does both in one move:

**Pattern:** `df.loc[rows, columns]` — the row condition first, the column list second.

In [ ]:
# Example 1 -- recent AND expensive
rows = (ames['Year Built'] > 2000) & (ames['SalePrice'] > 200000)
cols = ['Neighborhood', 'SalePrice', 'Gr Liv Area']

recent_expensive = ames.loc[rows, cols]
print(f"Matching homes: {recent_expensive.shape[0]}")
recent_expensive.head(3)

In [ ]:
# Example 2 -- either end of the market
rows = (ames['SalePrice'] < 100000) | (ames['SalePrice'] > 500000)
cols = ['Neighborhood', 'SalePrice', 'Gr Liv Area']

price_extremes = ames.loc[rows, cols]
print(f"Matching homes: {price_extremes.shape[0]}")
price_extremes.head(3)

Recognize that **254**? It's the same set of houses we filtered a moment ago — except now we're getting back just the three columns we asked for instead of all 82.

### Data Mystery #2 🕵️‍♀️

**Your challenge:** a developer wants to know where the market for **newer, larger homes** actually is.

Work these in order — each step builds on the one before it:

1. Use **`.loc[]`** to pull the `Neighborhood`, `Year Built`, `Gr Liv Area`, and `SalePrice` columns for homes built in **2000 or later** *and* **3,000 sq ft or larger**.
2. **How many homes** meet that condition?
3. For those homes: what's the **average sale price**, and **how many different neighborhoods** do they span?

**Detective tools:** `.loc[]` · `.shape` · `.mean()` · `.nunique()`

**Hint:** Save your `.loc[]` result to a variable — then steps 2 and 3 are one short line each.

In [ ]:
# Step 1: filter rows and select columns with .loc[]
# Your code here:



In [ ]:
# Step 2: how many homes meet the condition?
# Your code here:



In [ ]:
# Step 3: average sale price, and how many neighborhoods?
# Your code here:



### Common Detective Mistakes 🚨

Every one of these will bite you at some point. They're all quick to spot once you know the symptom.

| Mistake | Symptom | Fix |
|---|---|---|
| `ames.head` | Prints a function, not data | Add the parentheses: `ames.head()` |
| `ames['saleprice']` | `KeyError` | Column names are **case sensitive**: `ames['SalePrice']` |
| `ames[(cond1) and (cond2)]` | `ValueError` | Use `&` and `\|`, not `and` / `or` |
| `ames[Year Built > 2000]` | `NameError` | Reference the column: `ames[ames['Year Built'] > 2000]` |
| Confusing `[ ]` and `[[ ]]` | Wrong object type | `ames['col']` → Series, `ames[['col']]` → DataFrame |

**Pro tip:** when a column name won't work, run `ames.columns` and check the exact spelling and spacing.

## Putting It All Together

Remember the spreadsheet challenge from the start of class? Let's solve it the Python way.

**Your task:** find the **average number of seats** on aircraft manufactured by **Embraer** in **2004 or later**.

In [ ]:
# Load the planes dataset
planes_url = "https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/planes.csv"
planes = pd.read_csv(planes_url)

planes.head()

### Try It

Write the code to:

- Filter for **Embraer** aircraft (check the `manufacturer` column) built in **2004 or later**
- Calculate the **average number of seats**

**Hint:** `.loc[]` for the filtering, `.mean()` for the average.

In [ ]:
# Your code here


## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `pd.read_csv(url)` / `pd.read_csv(path)` | Reads a CSV into a DataFrame — copies it into memory **for this session only** |
| `df.shape` | How big is it? Rows and columns, as a tuple |
| `df.head()` | Peek at the first 5 rows |
| `df.columns` | Every column name — your lookup when you forget one |
| `df.dtypes` | The data type of each column |
| `df.info()` | Types *and* missing-value counts, in one summary |
| `df.describe()` | Summary statistics for the numeric columns |
| `df['col']` | One column, as a **Series** |
| `df[['col']]` | One column, as a **DataFrame** |
| `df[['a', 'b', 'c']]` | Several columns — the inner `[ ]` is just a Python list |
| `df[df['col'] > value]` | Keeps only the rows where the condition is `True` |
| `&` and `\|` | Combine conditions — `&` **narrows** (AND), `\|` **widens** (OR) |
| `df.loc[rows, cols]` | Filter rows **and** select columns in a single step |
| `s.mean()` · `s.max()` · `s.min()` | Collapse a column down to one number |
| `s.nunique()` | How many *distinct* values a column holds |

**The two ideas underneath all of it:**

1. **Know what you're holding.** One name in `[ ]` gives you a Series; a list in `[[ ]]` gives you a DataFrame — and they don't share the same toolkit.
2. **Subset first, then analyze.** Narrow to the rows and columns that matter, *then* ask your questions of that subset.

## What's Next

- **Thursday Lab:** hands-on practice importing multiple datasets, exploring messy real-world data, and building filtering workflows.
- **Chapters to review:** Chapter 7 (Importing Data), Chapter 8 (DataFrames), Chapter 9 (Subsetting) — read these after today's lecture and before Thursday's lab.